# CE-4 Project: Academic Affairs Dashboard
### Faculty Roster, Course Allocations & Lecture Log Dashboard
**Roll Number:** 25SSDS415013

This notebook builds a relational SQLite database and a Tkinter desktop dashboard. It contains four live views: a joined academic workload view and the three raw database tables.

## 1. Import required libraries
We use `sqlite3` for the database, `tkinter` for the GUI, and `datetime` for lecture dates. These are included with most standard Python installations.

In [9]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import date
from pathlib import Path

DB_FILE = Path("academic_affairs.db")
print("Libraries imported successfully.")
print("Database file:", DB_FILE.resolve())

Libraries imported successfully.
Database file: C:\Users\Charan Gowda B S\academic_affairs.db


## 2. Create the database tables
The three tables are `faculty`, `courses`, and `lecture_logs`. Foreign keys connect courses to faculty and lecture logs to courses. `ON DELETE CASCADE` automatically removes dependent rows when a parent row is deleted.

In [10]:
def get_connection():
    """Open SQLite connection and enable foreign-key enforcement."""
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_tables():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS faculty (
                faculty_id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL COLLATE NOCASE UNIQUE,
                department TEXT NOT NULL)
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS courses (
                course_id INTEGER PRIMARY KEY AUTOINCREMENT,
                faculty_id INTEGER NOT NULL,
                course_code TEXT NOT NULL,
                credits INTEGER NOT NULL DEFAULT 3,
                FOREIGN KEY (faculty_id) REFERENCES faculty(faculty_id) ON DELETE CASCADE,
                UNIQUE (faculty_id, course_code))
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS lecture_logs (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                course_id INTEGER NOT NULL,
                topic_covered TEXT NOT NULL,
                duration_hours REAL NOT NULL CHECK (duration_hours > 0),
                lecture_date TEXT NOT NULL,
                FOREIGN KEY (course_id) REFERENCES courses(course_id) ON DELETE CASCADE)
        """)
    print("Database and tables are ready.")

create_tables()

Database and tables are ready.


## 3. Insert sample records when the database is empty
Sample rows are inserted only if all three tables are empty. Re-running this cell will not duplicate the sample data.

In [11]:
def seed_database():
    with get_connection() as conn:
        count = conn.execute("SELECT COUNT(*) FROM faculty").fetchone()[0]
        if count == 0:
            faculty_rows = [
                ("Anita Rao", "Computer Science"),
                ("Ramesh Kulkarni", "Electronics"),
                ("Meera Nair", "Mathematics")
            ]
            conn.executemany(
                "INSERT INTO faculty (name, department) VALUES (?, ?)", faculty_rows
            )
            faculty_ids = {name: fid for fid, name in conn.execute(
                "SELECT faculty_id, name FROM faculty"
            ).fetchall()}
            course_rows = [
                (faculty_ids["Anita Rao"], "CS-301", 4),
                (faculty_ids["Ramesh Kulkarni"], "EEE-102", 3),
                (faculty_ids["Meera Nair"], "MA-201", 3)
            ]
            conn.executemany(
                "INSERT INTO courses (faculty_id, course_code, credits) VALUES (?, ?, ?)",
                course_rows
            )
            course_ids = {code: cid for cid, code in conn.execute(
                "SELECT course_id, course_code FROM courses"
            ).fetchall()}
            logs = [
                (course_ids["CS-301"], "Introduction to Data Structures", 1.5, str(date.today())),
                (course_ids["EEE-102"], "Basic Circuit Analysis", 1.0, str(date.today())),
                (course_ids["MA-201"], "Differential Calculus", 1.5, str(date.today()))
            ]
            conn.executemany(
                "INSERT INTO lecture_logs (course_id, topic_covered, duration_hours, lecture_date) VALUES (?, ?, ?, ?)",
                logs
            )
            print("Sample faculty, courses, and lecture logs inserted.")
        else:
            print("Existing database records found; sample data was not inserted again.")

seed_database()

Existing database records found; sample data was not inserted again.


## 4. Check the database contents
This quick check displays the number of records in each table.

In [12]:
with get_connection() as conn:
    for table in ("faculty", "courses", "lecture_logs"):
        total = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {total} record(s)")

faculty: 5 record(s)
courses: 5 record(s)
lecture_logs: 5 record(s)


## 5. Define database operations
The insert operation reuses an existing faculty profile when the name already exists. Otherwise, it creates a new faculty profile. It then creates a course allocation and its lecture log. Parameterized SQL protects the database from SQL injection.

In [13]:
def add_academic_record(name, department, course_code, topic, duration):
    with get_connection() as conn:
        row = conn.execute(
            "SELECT faculty_id FROM faculty WHERE name = ?", (name,)
        ).fetchone()
        if row:
            faculty_id = row[0]
            # Update department if the user supplied a different department.
            conn.execute("UPDATE faculty SET department = ? WHERE faculty_id = ?",
                         (department, faculty_id))
        else:
            cur = conn.execute(
                "INSERT INTO faculty (name, department) VALUES (?, ?)",
                (name, department)
            )
            faculty_id = cur.lastrowid

        course = conn.execute(
            "SELECT course_id FROM courses WHERE faculty_id = ? AND course_code = ?",
            (faculty_id, course_code)
        ).fetchone()
        if course:
            course_id = course[0]
        else:
            cur = conn.execute(
                "INSERT INTO courses (faculty_id, course_code, credits) VALUES (?, ?, 3)",
                (faculty_id, course_code)
            )
            course_id = cur.lastrowid

        conn.execute(
            "INSERT INTO lecture_logs (course_id, topic_covered, duration_hours, lecture_date) VALUES (?, ?, ?, ?)",
            (course_id, topic, duration, str(date.today()))
        )


def get_table_rows(table_name):
    allowed = {"faculty", "courses", "lecture_logs"}
    if table_name not in allowed:
        raise ValueError("Invalid table name")
    with get_connection() as conn:
        return conn.execute(f"SELECT * FROM {table_name} ORDER BY 1").fetchall()


def get_join_rows():
    sql = """
        SELECT f.name, f.department, c.course_code, l.topic_covered,
               l.duration_hours, l.lecture_date, c.course_id, l.log_id
        FROM faculty AS f
        JOIN courses AS c ON f.faculty_id = c.faculty_id
        JOIN lecture_logs AS l ON c.course_id = l.course_id
        ORDER BY f.name, c.course_code, l.log_id
    """
    with get_connection() as conn:
        return conn.execute(sql).fetchall()


def get_delete_options(target):
    queries = {
        "Faculty": ("SELECT faculty_id, name FROM faculty ORDER BY name",),
        "Course": ("SELECT c.course_id, c.course_code || ' — ' || f.name FROM courses c JOIN faculty f ON c.faculty_id=f.faculty_id ORDER BY c.course_id",),
        "Lecture Log": ("SELECT l.log_id, l.topic_covered || ' — ' || f.name FROM lecture_logs l JOIN courses c ON l.course_id=c.course_id JOIN faculty f ON c.faculty_id=f.faculty_id ORDER BY l.log_id",)
    }
    with get_connection() as conn:
        return conn.execute(queries[target][0]).fetchall()


def delete_record(target, record_id):
    table_and_key = {
        "Faculty": ("faculty", "faculty_id"),
        "Course": ("courses", "course_id"),
        "Lecture Log": ("lecture_logs", "log_id")
    }
    table, key = table_and_key[target]
    with get_connection() as conn:
        conn.execute(f"DELETE FROM {table} WHERE {key} = ?", (record_id,))

print("Database operations are ready.")

Database operations are ready.


## 6. Build the Tkinter dashboard
The dashboard includes:
- An insert form for faculty name, department, course code, topic, and lecture duration.
- A delete form with a target type and dynamically populated record selector.
- Four tabs: joined workload, faculty, courses, and lecture logs.
- Automatic refresh and input clearing after successful changes.

**Note:** Tkinter opens a separate desktop window. 

In [14]:
class AcademicDashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("Academic Affairs — Faculty & Lecture Dashboard")
        self.root.geometry("1180x760")
        self.root.minsize(900, 600)

        title = ttk.Label(root, text="ACADEMIC AFFAIRS DASHBOARD", font=("Arial", 18, "bold"))
        title.pack(pady=10)
        subtitle = ttk.Label(root, text="Faculty Roster • Course Allocations • Lecture Logs")
        subtitle.pack(pady=(0, 10))

        form = ttk.LabelFrame(root, text="Operations Insert Panel", padding=10)
        form.pack(fill="x", padx=12, pady=6)
        self.name_var = tk.StringVar()
        self.dept_var = tk.StringVar()
        self.course_var = tk.StringVar()
        self.topic_var = tk.StringVar()
        self.duration_var = tk.StringVar()
        fields = [
            ("Faculty Name", self.name_var),
            ("Department", self.dept_var),
            ("Course Code", self.course_var),
            ("Topic Covered", self.topic_var),
            ("Duration (hours)", self.duration_var)
        ]
        for col, (label, variable) in enumerate(fields):
            ttk.Label(form, text=label).grid(row=0, column=col, padx=5, sticky="w")
            ttk.Entry(form, textvariable=variable, width=20).grid(row=1, column=col, padx=5, pady=5, sticky="ew")
        ttk.Button(form, text="Add Record", command=self.add_record).grid(row=1, column=len(fields), padx=8)

        delete_frame = ttk.LabelFrame(root, text="Operations Delete Panel", padding=10)
        delete_frame.pack(fill="x", padx=12, pady=6)
        self.delete_type = tk.StringVar(value="Faculty")
        self.delete_choice = tk.StringVar()
        ttk.Label(delete_frame, text="Delete target").grid(row=0, column=0, padx=5)
        self.type_box = ttk.Combobox(delete_frame, textvariable=self.delete_type,
                                     values=["Faculty", "Course", "Lecture Log"],
                                     state="readonly", width=18)
        self.type_box.grid(row=0, column=1, padx=5)
        self.type_box.bind("<<ComboboxSelected>>", lambda event: self.refresh_delete_options())
        ttk.Label(delete_frame, text="Select record").grid(row=0, column=2, padx=5)
        self.choice_box = ttk.Combobox(delete_frame, textvariable=self.delete_choice,
                                       state="readonly", width=55)
        self.choice_box.grid(row=0, column=3, padx=5)
        ttk.Button(delete_frame, text="Delete Selected", command=self.delete_selected).grid(row=0, column=4, padx=8)

        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True, padx=12, pady=8)
        self.tree_configs = {
            "Master Academic Workload Desk": ("join", ["Faculty Name", "Department", "Course Code", "Topic Covered", "Duration Hours", "Lecture Date"]),
            "Faculty": ("faculty", ["Faculty ID", "Name", "Department"]),
            "Courses": ("courses", ["Course ID", "Faculty ID", "Course Code", "Credits"]),
            "Lecture Logs": ("lecture_logs", ["Log ID", "Course ID", "Topic Covered", "Duration Hours", "Lecture Date"])
        }
        self.trees = {}
        for tab_name, (key, columns) in self.tree_configs.items():
            frame = ttk.Frame(self.notebook)
            self.notebook.add(frame, text=tab_name)
            tree = ttk.Treeview(frame, columns=columns, show="headings", height=15)
            yscroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
            xscroll = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
            tree.configure(yscrollcommand=yscroll.set, xscrollcommand=xscroll.set)
            for column in columns:
                tree.heading(column, text=column)
                tree.column(column, width=150, minwidth=100, anchor="center")
            tree.grid(row=0, column=0, sticky="nsew")
            yscroll.grid(row=0, column=1, sticky="ns")
            xscroll.grid(row=1, column=0, sticky="ew")
            frame.rowconfigure(0, weight=1)
            frame.columnconfigure(0, weight=1)
            self.trees[key] = tree

        self.status = tk.StringVar(value="Ready")
        ttk.Label(root, textvariable=self.status, relief="sunken", anchor="w").pack(fill="x", side="bottom")
        self.refresh_all()

    def add_record(self):
        name = self.name_var.get().strip()
        department = self.dept_var.get().strip()
        course = self.course_var.get().strip().upper()
        topic = self.topic_var.get().strip()
        duration_text = self.duration_var.get().strip()
        if not all([name, department, course, topic, duration_text]):
            messagebox.showwarning("Missing information", "Please fill in every field before adding a record.")
            return
        try:
            duration = float(duration_text)
            if duration <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid duration", "Duration must be a positive numeric value, such as 1.5.")
            return
        try:
            add_academic_record(name, department, course, topic, duration)
            for variable in [self.name_var, self.dept_var, self.course_var, self.topic_var, self.duration_var]:
                variable.set("")
            self.refresh_all()
            self.status.set("Record added successfully.")
            messagebox.showinfo("Success", "Faculty/course/lecture record saved successfully.")
        except sqlite3.IntegrityError as exc:
            messagebox.showerror("Database error", str(exc))
        except Exception as exc:
            messagebox.showerror("Error", str(exc))

    def refresh_delete_options(self):
        target = self.delete_type.get()
        self.delete_items = get_delete_options(target)
        self.choice_box["values"] = [f"{label} (ID: {record_id})" for record_id, label in self.delete_items]
        self.delete_choice.set("")

    def delete_selected(self):
        target = self.delete_type.get()
        selected = self.delete_choice.get()
        if not selected:
            messagebox.showwarning("Select a record", "Choose a record from the dropdown first.")
            return
        index = self.choice_box.current()
        if index < 0 or index >= len(self.delete_items):
            messagebox.showwarning("Invalid selection", "Please select a current record from the dropdown.")
            self.refresh_delete_options()
            return
        record_id = self.delete_items[index][0]
        if not messagebox.askyesno("Confirm deletion", f"Delete selected {target.lower()} record? Related child rows may also be deleted."):
            return
        try:
            delete_record(target, record_id)
            self.refresh_all()
            self.refresh_delete_options()
            self.status.set(f"Selected {target.lower()} record deleted.")
            messagebox.showinfo("Deleted", "The selected record was deleted and the views were refreshed.")
        except Exception as exc:
            messagebox.showerror("Delete error", str(exc))

    def refresh_all(self):
        for key, tree in self.trees.items():
            for item in tree.get_children():
                tree.delete(item)
            if key == "join":
                rows = [row[:6] for row in get_join_rows()]
            else:
                rows = get_table_rows(key)
            for row in rows:
                tree.insert("", "end", values=row)
        self.refresh_delete_options()


def launch_dashboard():
    try:
        root = tk.Tk()
    except tk.TclError as exc:
        print("Tkinter could not open a desktop window in this environment.")
        print("Run this notebook on your local computer with a graphical desktop, or run the same code as a .py script.")
        print("System message:", exc)
        return None
    app = AcademicDashboard(root)
    root.mainloop()
    return app

print("Dashboard interface is defined. Run the next cell to open it.")

Dashboard interface is defined. Run the next cell to open it.


## 7. Launch the dashboard
Run this cell to open the desktop application. Keep the window open while using the dashboard. Close the window to return to the notebook.

In [ ]:
dashboard = launch_dashboard()

## 8. Verify cascading deletes (optional database check)
The dashboard's delete panel demonstrates cascading deletes. This final check shows the foreign-key definitions configured in SQLite.

In [ ]:
with get_connection() as conn:
    print("Foreign keys on courses:")
    for row in conn.execute("PRAGMA foreign_key_list(courses)"):
        print(row)
    print("\nForeign keys on lecture_logs:")
    for row in conn.execute("PRAGMA foreign_key_list(lecture_logs)"):
        print(row)

print("\nProject execution completed. Database file:", DB_FILE.resolve())

## 9. Project interpretation and conclusion

### Interpretation
- The `faculty` table stores each instructor once.
- The `courses` table links each course allocation to a faculty member through `faculty_id`.
- The `lecture_logs` table links each lecture record to a course through `course_id`.
- The Master Academic Workload Desk uses SQL `JOIN` operations to display related information from all three tables.
- SQLite foreign-key enforcement and `ON DELETE CASCADE` prevent orphaned course and lecture records.
- The GUI validates required text fields and accepts only positive numeric lecture durations.
- After a successful insertion or deletion, the input fields are cleared and all four data views are refreshed.

### Conclusion
This project demonstrates how Python, Tkinter, and SQLite can be integrated to build a relational academic workload management system. The application supports faculty creation and reuse, course allocation, lecture logging, joined reporting, record deletion, data validation, and cascading deletion of dependent records.